# 01 — Build the FDA novel-approval event list (2015–2025)

**Goal:** collect every CDER novel drug approval, attach the sponsor company, flag obvious large caps, and export a sheet for manual review.

**Why automate:** ~500 drugs is too many to type by hand, and a script is *reproducible*: anyone can rerun it and get the same raw list. That matters for credibility.

Run cells top to bottom with Shift+Enter.

In [1]:
import io, re, time
import requests
import pandas as pd

HEADERS = {"User-Agent": "Mozilla/5.0 (student research project)"}
YEARS = range(2015, 2026)
URL_PATTERNS = [
    "https://www.fda.gov/drugs/novel-drug-approvals-fda/novel-drug-approvals-{year}",
    "https://www.fda.gov/drugs/new-drugs-fda-cders-new-molecular-entities-and-new-therapeutic-biological-products/novel-drug-approvals-{year}",
]

## Step 1: Scrape the FDA yearly tables
FDA has used different URLs over the years, so we try two patterns. `pd.read_html` turns every HTML table on the page into a DataFrame, and we keep the one with a drug name and a date column.

In [2]:
def fetch_year(year):
    for pattern in URL_PATTERNS:
        url = pattern.format(year=year)
        try:
            r = requests.get(url, headers=HEADERS, timeout=30)
            if r.status_code != 200:
                continue
            tables = pd.read_html(io.StringIO(r.text))
        except (requests.RequestException, ValueError):
            continue
        for t in tables:
            cols = [str(c).lower() for c in t.columns]
            if any("drug name" in c for c in cols) and any("date" in c for c in cols):
                t = t.copy()
                t["fda_year"] = year
                t["source_url"] = url
                return t
    return None

def standardize(t):
    rename = {}
    for c in t.columns:
        lc = str(c).lower()
        if "drug name" in lc:
            rename[c] = "drug_name"
        elif "active ingredient" in lc:
            rename[c] = "active_ingredient"
        elif "date" in lc:
            rename[c] = "fda_approval_date"
        elif "approved use" in lc:
            rename[c] = "approved_use"
    t = t.rename(columns=rename)
    keep = ["drug_name", "active_ingredient", "fda_approval_date", "approved_use", "fda_year", "source_url"]
    return t[[c for c in keep if c in t.columns]]

frames, failed = [], []
for y in YEARS:
    t = fetch_year(y)
    if t is None:
        failed.append(y)
        print(f"{y}: FAILED - handle manually")
    else:
        frames.append(standardize(t))
        print(f"{y}: {len(t)} approvals")
    time.sleep(1)  # be polite to fda.gov

events = pd.concat(frames, ignore_index=True)
print("Total:", len(events), "| Failed years:", failed)

2015: FAILED - handle manually
2016: FAILED - handle manually
2017: FAILED - handle manually
2018: FAILED - handle manually
2019: FAILED - handle manually
2020: FAILED - handle manually
2021: 50 approvals
2022: 37 approvals
2023: 55 approvals
2024: 50 approvals
2025: 46 approvals
Total: 238 | Failed years: [2015, 2016, 2017, 2018, 2019, 2020]


In [8]:
path = "../data/raw/fda_nme_compilation_1985_2025.xlsx"
print(pd.ExcelFile(path).sheet_names)

nme = pd.read_excel(path)
print(nme.shape)
print(list(nme.columns))
nme.head()

['2025 Compilation']
(1387, 27)
['Proprietary  Name', 'Active Ingredient/Moiety', 'Applicant', 'NDA/BLA', ' Application Number(1)', ' Application Number(2)', ' Application Number(3)', 'Dosage Form(1)', 'Route of Administration(1)', 'Dosage Form(2)', 'Route of Administration(2)', 'Dosage Form(3)', 'Route of Administration(3)', 'FDA Receipt Date', 'FDA Approval Date', 'Approval Year', 'Abbreviated Indication(s)', 'Approved Use(s)', 'Review Designation', 'Orphan Drug Designation', 'Accelerated Approval', 'Breakthrough Therapy Designation', 'Fast Track Designation', 'Qualified Infectious Disease Product', 'Issued a Priority Review Voucher', 'Redeemed a Priority Review Voucher', 'Notes']


,Proprietary Name,Active Ingredient/Moiety,Applicant,NDA/BLA,Application Number(1),Application Number(2),Application Number(3),Dosage Form(1),Route of Administration(1),Dosage Form(2),...,Approved Use(s),Review Designation,Orphan Drug Designation,Accelerated Approval,Breakthrough Therapy Designation,Fast Track Designation,Qualified Infectious Disease Product,Issued a Priority Review Voucher,Redeemed a Priority Review Voucher,Notes
0,Lupron,leuprolide acetate,TAP Pharmaceuticals,NDA,19010,NaN,NaN,Injectable,Injection,NaN,...,NaN,Priority,No,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Seldane,terfenadine,Merrell-Dow,NDA,18949,NaN,NaN,Tablet,Oral,NaN,...,NaN,Priority,No,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Ridaura,auranofin,SmithKline & French,NDA,18689,NaN,NaN,Capsule,Oral,NaN,...,NaN,Priority,No,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Marinol,dronabinol,Unimed,NDA,18651,NaN,NaN,Capsule,Oral,NaN,...,NaN,Priority,No,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Fortaz,ceftazidime,Glaxo,NDA,50578,NaN,NaN,Injectable,Injection,NaN,...,NaN,Priority,No,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [14]:
nme.columns = nme.columns.str.strip().str.replace(r"\s+", " ", regex=True)
nme["FDA Approval Date"] = pd.to_datetime(nme["FDA Approval Date"], errors="coerce")

sample = nme[nme["Approval Year"].between(2015,2025)].copy()
print(len(sample), "approvals from 2015-2025")
print(sample.groupby("Approval Year").size())
sample.to_csv("../data/processed/nme_2015_2025.csv", index=False)

511 approvals from 2015-2025
Approval Year
2015    45
2016    22
2017    46
2018    59
2019    48
2020    53
2021    50
2022    37
2023    55
2024    50
2025    46
dtype: int64


## Step 2: Clean and sanity-check
**Always check scraped data against a known number.** FDA's published totals include 22 (2016), 59 (2018), 55 (2023), and 50 (2024). If your counts don't match, something broke.

In [ ]:
events["drug_name"] = events["drug_name"].astype(str).str.replace(r"[\*†‡]", "", regex=True).str.strip()
if "approved_use" in events:
    events["approved_use"] = events["approved_use"].astype(str).str.replace("Drug Trials Snapshot", "").str.strip()
events["fda_approval_date"] = pd.to_datetime(events["fda_approval_date"], errors="coerce")

print(events.groupby("fda_year").size())
print("Unparseable dates:", events["fda_approval_date"].isna().sum())

events.to_csv("../data/raw/fda_novel_approvals_2015_2025.csv", index=False)
events.head()

## Step 3: Look up sponsors with openFDA
openFDA is the FDA's free public API. We search Drugs@FDA by brand name and prefer NDA/BLA applications (the original new-drug filings).

⚠️ **Caveat to remember for interviews:** `sponsor_name` is the *current* application holder. If a small biotech was acquired after approval, this may show the acquirer. You'll confirm the company **at the approval date** from its press release.

In [ ]:
def lookup_sponsor(brand):
    name = re.sub(r"[^A-Za-z0-9 \-]", "", str(brand)).strip()
    params = {"search": f'products.brand_name:"{name}"', "limit": 5}
    try:
        r = requests.get("https://api.fda.gov/drug/drugsfda.json", params=params, timeout=30)
        if r.status_code != 200:
            return None, None
        results = r.json().get("results", [])
    except (requests.RequestException, ValueError):
        return None, None
    for app in results:
        num = app.get("application_number", "")
        if num.startswith(("NDA", "BLA")):
            return app.get("sponsor_name"), num
    if results:
        return results[0].get("sponsor_name"), results[0].get("application_number")
    return None, None

sponsors, apps = [], []
for i, brand in enumerate(events["drug_name"]):
    s, a = lookup_sponsor(brand)
    sponsors.append(s)
    apps.append(a)
    if i % 25 == 0:
        print(f"{i}/{len(events)}")
    time.sleep(0.3)  # openFDA allows ~240 requests/min without a key

events["openfda_sponsor"] = sponsors
events["application_number"] = apps
print("No sponsor found:", events["openfda_sponsor"].isna().sum())

## Step 4: Flag obvious large caps
This only **flags**; nothing is deleted. You still glance at every row, because a small biotech's drug can be listed under a big partner's name (or vice versa).

In [ ]:
LARGE_CAP = ["pfizer", "merck", "lilly", "abbvie", "janssen", "johnson", "bristol", "squibb",
             "novartis", "genentech", "roche", "astrazeneca", "glaxo", "gsk", "sanofi",
             "novo nordisk", "bayer", "takeda", "amgen", "gilead", "boehringer", "astellas",
             "daiichi", "eisai", "otsuka", "biogen", "regeneron", "vertex", "shionogi"]

def screen(sponsor):
    if not isinstance(sponsor, str):
        return "review - sponsor unknown"
    s = sponsor.lower()
    return "likely_exclude_large_cap" if any(k in s for k in LARGE_CAP) else "review"

events["screen"] = events["openfda_sponsor"].apply(screen)
print(events["screen"].value_counts())

## Step 5: Export the manual-review sheet
Open this in Google Sheets or Excel and fill in the blank columns for every `review` row. Rows flagged large-cap just need a quick glance and `include = N`, `exclude_reason = large cap`.

In [ ]:
manual_cols = ["company_at_approval", "ticker", "exchange", "approx_market_cap_usd",
               "press_release_datetime_et", "timing", "event_day0",
               "priority_review", "breakthrough", "orphan", "first_in_class",
               "confounding_event", "include", "exclude_reason", "notes"]
for c in manual_cols:
    events[c] = ""

events = events.sort_values(["screen", "fda_approval_date"], ascending=[False, False])
events.to_csv("../data/processed/event_review_sheet.csv", index=False)
print("Saved. Rows needing review:", (events["screen"] != "likely_exclude_large_cap").sum())